# Backfill Missing Raw Sources

Точечно дозагружает raw-исходники, которые нужны для закрытия найденных пропусков:

- `raw/klines/ADAUSDT/1m/2020-02-01` - первая минута `00:00` для ADA-фичей;
- `raw/klines/BTCUSDT/1m/2020-02-01` - первая минута `00:00` для `btc_features`;
- `raw/klines/ADAUSDT/1m/2026-02-02` - следующая минута для `return_1m_forward` на `2026-02-01 23:59`.

После дозагрузки raw нужно пересобрать производные feature-партиции за затронутые даты. Для `intraminute_dynamics` отдельной дозагрузки raw не требуется: исходники уже есть, нужно только пересобрать отсутствующие feature-дни `2020-03-15`, `2022-04-19`, `2022-05-01`.

In [1]:
import io
import os
import time
import zipfile
from pathlib import Path

import boto3
import pandas as pd
import requests
from dotenv import load_dotenv

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## Settings

In [2]:
BUCKET = "binance-data-downloader"
RAW_PREFIX = "raw"
FEATURE_SYMBOL = "ADAUSDT"
SYMBOL = FEATURE_SYMBOL
INTERVAL = "1m"
BASE_ROOT = "https://data.binance.vision/data/futures/um/daily"

# 2020-02-01 already exists in S3, but it is incomplete, so overwrite is needed.
FORCE_OVERWRITE_EXISTING = True

KLINES_BACKFILL_TASKS = [
    {"symbol": "ADAUSDT", "date": "2020-02-01", "reason": "missing 00:00 in ADAUSDT klines backbone"},
    {"symbol": "BTCUSDT", "date": "2020-02-01", "reason": "missing 00:00 in BTCUSDT klines backbone"},
    {"symbol": "ADAUSDT", "date": "2026-02-02", "reason": "needed for ADAUSDT return_1m_forward at 2026-02-01 23:59"},
]

INTRAMINUTE_DYNAMICS_DAYS_TO_REBUILD = ["2020-03-15", "2022-04-19", "2022-05-01"]

In [3]:
load_dotenv(dotenv_path=".env")

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("YC_ENDPOINT"),
    region_name=os.getenv("YC_REGION"),
    aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
)

## Klines Download Helpers

Здесь чтение CSV сделано осторожнее, чем в старом `BinanceClient`: старые Binance daily-файлы иногда идут без header-строки, и чтение с `header=0` может удалить первую минуту.

In [4]:
KLINES_COLUMNS = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time",
    "quote_volume",
    "trades",
    "taker_buy_base",
    "taker_buy_quote",
    "ignore",
]

FINAL_KLINES_COLUMNS = [
    "timestamp",
    "open_time",
    "close_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "quote_volume",
    "trades",
    "taker_buy_base",
    "taker_buy_quote",
]


def klines_url(symbol, date, interval=INTERVAL):
    return f"{BASE_ROOT}/klines/{symbol}/{interval}/{symbol}-{interval}-{date}.zip"


def klines_key(symbol, date, raw_prefix=RAW_PREFIX, interval=INTERVAL):
    return f"{raw_prefix.strip('/')}/klines/symbol={symbol}/interval={interval}/date={date}/data.parquet"


def s3_key_exists(key):
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
        return True
    except Exception as exc:
        code = getattr(exc, "response", {}).get("Error", {}).get("Code")
        if code in {"404", "NoSuchKey", "NotFound"}:
            return False
        raise


def download_zip(url, retries=3, timeout=(10, 60), backoff=2):
    for attempt in range(1, retries + 1):
        response = requests.get(url, timeout=timeout)
        if response.status_code == 200:
            return response.content
        if response.status_code == 404:
            return None
        if attempt == retries:
            response.raise_for_status()
        time.sleep(backoff ** attempt)
    return None


def read_binance_zip_csv(content):
    with zipfile.ZipFile(io.BytesIO(content)) as zf:
        names = [name for name in zf.namelist() if not name.endswith("/")]
        if not names:
            raise ValueError("ZIP has no files")
        with zf.open(names[0]) as fh:
            df = pd.read_csv(fh, header=None, low_memory=False)

    df = df.dropna(how="all")
    if df.empty:
        return df

    # Drop a real header row if Binance included one.
    first_open_time = pd.to_numeric(pd.Series([df.iloc[0, 0]]), errors="coerce").iloc[0]
    if pd.isna(first_open_time):
        df = df.iloc[1:].reset_index(drop=True)

    return df


def normalize_klines(df):
    if df.shape[1] < 11:
        raise ValueError(f"Klines source has {df.shape[1]} columns, expected at least 11")

    out = df.iloc[:, : min(df.shape[1], len(KLINES_COLUMNS))].copy()
    out.columns = KLINES_COLUMNS[: out.shape[1]]

    float_cols = ["open", "high", "low", "close", "volume", "quote_volume", "taker_buy_base", "taker_buy_quote"]
    int_cols = ["open_time", "close_time", "trades"]

    for col in float_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("float32")
    for col in int_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")

    out = out.dropna(subset=["open_time", "close_time", "open", "high", "low", "close"])
    for col in int_cols:
        out[col] = out[col].astype("int64")

    out["timestamp"] = pd.to_datetime(out["open_time"], unit="ms", utc=True)
    out = out.drop_duplicates(subset=["open_time"]).sort_values("open_time").reset_index(drop=True)
    return out[FINAL_KLINES_COLUMNS]


def expected_minutes(date):
    return pd.date_range(pd.Timestamp(date, tz="UTC"), periods=1440, freq="min")


def extract_minute_index(df):
    if "timestamp" in df.columns:
        timestamps = pd.to_datetime(df["timestamp"], utc=True)
    elif "open_time" in df.columns:
        timestamps = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    else:
        raise KeyError(f"Expected timestamp or open_time column, got: {list(df.columns)}")

    return pd.DatetimeIndex(timestamps.dt.floor("min").dropna().unique()).sort_values()


def minute_coverage(df, date):
    actual = extract_minute_index(df)
    expected = expected_minutes(date)
    return {
        "rows": len(df),
        "unique_minutes": len(actual),
        "missing_minutes": list(expected.difference(actual)),
        "extra_minutes": list(actual.difference(expected)),
    }


def upload_parquet(df, key):
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow", compression="zstd")
    s3.put_object(Bucket=BUCKET, Key=key, Body=buffer.getvalue())

## Download And Upload Raw Klines

In [5]:
download_results = []

for task in KLINES_BACKFILL_TASKS:
    symbol = task["symbol"]
    date = task["date"]
    key = klines_key(symbol, date)
    url = klines_url(symbol, date)
    exists_before = s3_key_exists(key)

    if exists_before and not FORCE_OVERWRITE_EXISTING:
        download_results.append({**task, "key": key, "status": "skipped_exists"})
        continue

    print(f"download {url}")
    content = download_zip(url)
    if content is None:
        download_results.append({**task, "key": key, "status": "no_source_zip"})
        continue

    raw_csv = read_binance_zip_csv(content)
    normalized = normalize_klines(raw_csv)
    coverage = minute_coverage(normalized, date)

    upload_parquet(normalized, key)
    download_results.append(
        {
            **task,
            "key": key,
            "exists_before": exists_before,
            "status": "uploaded",
            "rows": coverage["rows"],
            "unique_minutes": coverage["unique_minutes"],
            "missing_minutes_count": len(coverage["missing_minutes"]),
            "missing_minutes": [str(x) for x in coverage["missing_minutes"][:20]],
            "extra_minutes_count": len(coverage["extra_minutes"]),
            "extra_minutes": [str(x) for x in coverage["extra_minutes"][:20]],
        }
    )

download_results_df = pd.DataFrame(download_results)
download_results_df

download https://data.binance.vision/data/futures/um/daily/klines/ADAUSDT/1m/ADAUSDT-1m-2020-02-01.zip
download https://data.binance.vision/data/futures/um/daily/klines/BTCUSDT/1m/BTCUSDT-1m-2020-02-01.zip
download https://data.binance.vision/data/futures/um/daily/klines/ADAUSDT/1m/ADAUSDT-1m-2026-02-02.zip


,symbol,date,reason,key,exists_before,status,rows,unique_minutes,missing_minutes_count,missing_minutes,extra_minutes_count,extra_minutes
0,ADAUSDT,2020-02-01,missing 00:00 in ADAUSDT klines backbone,raw/klines/symbol=ADAUSDT/interval=1m/date=202...,True,uploaded,1440,1440,0,[],0,[]
1,BTCUSDT,2020-02-01,missing 00:00 in BTCUSDT klines backbone,raw/klines/symbol=BTCUSDT/interval=1m/date=202...,True,uploaded,1440,1440,0,[],0,[]
2,ADAUSDT,2026-02-02,needed for ADAUSDT return_1m_forward at 2026-0...,raw/klines/symbol=ADAUSDT/interval=1m/date=202...,False,uploaded,1440,1440,0,[],0,[]


## Verify Uploaded Raw Klines

In [6]:
verify_rows = []

for task in KLINES_BACKFILL_TASKS:
    symbol = task["symbol"]
    date = task["date"]
    key = klines_key(symbol, date)
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    coverage = minute_coverage(df, date)
    verify_rows.append(
        {
            "symbol": symbol,
            "date": date,
            "key": key,
            "rows": coverage["rows"],
            "unique_minutes": coverage["unique_minutes"],
            "missing_minutes_count": len(coverage["missing_minutes"]),
            "missing_minutes": [str(x) for x in coverage["missing_minutes"][:20]],
            "extra_minutes_count": len(coverage["extra_minutes"]),
            "extra_minutes": [str(x) for x in coverage["extra_minutes"][:20]],
        }
    )

verify_df = pd.DataFrame(verify_rows)
verify_df

,symbol,date,key,rows,unique_minutes,missing_minutes_count,missing_minutes,extra_minutes_count,extra_minutes
0,ADAUSDT,2020-02-01,raw/klines/symbol=ADAUSDT/interval=1m/date=202...,1440,1440,0,[],0,[]
1,BTCUSDT,2020-02-01,raw/klines/symbol=BTCUSDT/interval=1m/date=202...,1440,1440,0,[],0,[]
2,ADAUSDT,2026-02-02,raw/klines/symbol=ADAUSDT/interval=1m/date=202...,1440,1440,0,[],0,[]


## Save Backfill Report

In [7]:
report_dir = Path("coverage_reports")
report_dir.mkdir(exist_ok=True)

download_results_df.to_csv(report_dir / "raw_klines_backfill_results.csv", index=False)
verify_df.to_csv(report_dir / "raw_klines_backfill_verify.csv", index=False)

print(f"saved: {report_dir / 'raw_klines_backfill_results.csv'}")
print(f"saved: {report_dir / 'raw_klines_backfill_verify.csv'}")

saved: coverage_reports\raw_klines_backfill_results.csv
saved: coverage_reports\raw_klines_backfill_verify.csv


## Next Feature Rebuilds

Raw-исходники после этого будут закрыты. Дальше нужно пересобрать feature-партиции:

- за `2020-02-01`: все feature-группы, которые используют `klines` backbone, плюс `btc_features`;
- за `2026-02-01`: `return_1m_forward`, теперь с использованием `raw/klines/ADAUSDT/2026-02-02`;
- за `2020-03-15`, `2022-04-19`, `2022-05-01`: только `intraminute_dynamics`.

Этот блок просто фиксирует список, чтобы его не держать в голове.

In [8]:
feature_rebuild_plan = pd.DataFrame(
    [
        {"feature": "aggression_features", "dates": ["2020-02-01"], "reason": "uses ADAUSDT klines backbone"},
        {"feature": "btc_features", "dates": ["2020-02-01"], "reason": "uses BTCUSDT klines"},
        {"feature": "intraminute_features", "dates": ["2020-02-01"], "reason": "uses ADAUSDT klines"},
        {"feature": "intraminute_segments", "dates": ["2020-02-01"], "reason": "uses ADAUSDT klines backbone"},
        {"feature": "price_pressure", "dates": ["2020-02-01"], "reason": "uses ADAUSDT klines/trades minute alignment"},
        {"feature": "trade_distribution", "dates": ["2020-02-01"], "reason": "uses ADAUSDT klines backbone"},
        {"feature": "trades_minute_level", "dates": ["2020-02-01"], "reason": "uses ADAUSDT klines backbone"},
        {"feature": "return_1m_forward", "dates": ["2020-02-01", "2026-02-01"], "reason": "target depends on exact next minute close"},
        {"feature": "intraminute_dynamics", "dates": ["2020-02-01", *INTRAMINUTE_DYNAMICS_DAYS_TO_REBUILD], "reason": "depends on segments plus raw aggTrades/klines"},
    ]
)
feature_rebuild_plan

,feature,dates,reason
0,aggression_features,[2020-02-01],uses ADAUSDT klines backbone
1,btc_features,[2020-02-01],uses BTCUSDT klines
2,intraminute_features,[2020-02-01],uses ADAUSDT klines
3,intraminute_segments,[2020-02-01],uses ADAUSDT klines backbone
4,price_pressure,[2020-02-01],uses ADAUSDT klines/trades minute alignment
5,trade_distribution,[2020-02-01],uses ADAUSDT klines backbone
6,trades_minute_level,[2020-02-01],uses ADAUSDT klines backbone
7,return_1m_forward,"[2020-02-01, 2026-02-01]",target depends on exact next minute close
8,intraminute_dynamics,"[2020-02-01, 2020-03-15, 2022-04-19, 2022-05-01]",depends on segments plus raw aggTrades/klines


## Rebuild Affected Feature Partitions

???? ???? ??????????? ????? raw backfill. ?? ??????? ???????????? ? ?????????????? ?????? ?? feature-????????, ??????? ???????? ?? ??????????????? raw-????? ??? ???? ????????? ?????????.

??? `btc_features` ? `price_pressure` ? ??????? ??????????? ??? ???????? build-?????????/??????, ??????? ??? ????????? ? ????????? ?????? `unsupported_feature_rebuilds` ? ??????? ????????? ???? ?????????.

In [5]:
import json
from pathlib import Path


def load_notebook_namespace(notebook_path, code_cell_indexes):
    notebook = json.loads(Path(notebook_path).read_text(encoding="utf-8"))
    namespace = {"__name__": f"loaded_{Path(notebook_path).stem}"}
    for idx in code_cell_indexes:
        source = "".join(notebook["cells"][idx]["source"])
        exec(compile(source, f"{notebook_path}:cell:{idx}", "exec"), namespace)
    return namespace


def write_feature_partition(df, feature_name, date, symbol=FEATURE_SYMBOL, interval=INTERVAL):
    key = f"features/{feature_name}/symbol={symbol}/interval={interval}/date={date}/data.parquet"
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow", compression="zstd")
    s3.put_object(Bucket=BUCKET, Key=key, Body=buffer.getvalue())
    return key


def validate_feature_minutes(feature_name, date, symbol=FEATURE_SYMBOL, interval=INTERVAL):
    key = f"features/{feature_name}/symbol={symbol}/interval={interval}/date={date}/data.parquet"
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    coverage = minute_coverage(df, date)
    return {
        "feature": feature_name,
        "date": date,
        "key": key,
        "rows": coverage["rows"],
        "unique_minutes": coverage["unique_minutes"],
        "missing_minutes_count": len(coverage["missing_minutes"]),
        "missing_minutes": [str(x) for x in coverage["missing_minutes"][:20]],
        "extra_minutes_count": len(coverage["extra_minutes"]),
        "extra_minutes": [str(x) for x in coverage["extra_minutes"][:20]],
    }

### Load Existing Builders

??????????? ?????? ?????? ? imports/constants/functions. ?????? ? ???????? ??????? ?? ????? ????????? ?? ???????????.

In [6]:
builders = {
    "aggression_features": load_notebook_namespace("build_aggression_features.ipynb", [1, 2, 3, 4]),
    "intraminute_features": load_notebook_namespace("build_intraminute_features.ipynb", [1, 2, 3]),
    "intraminute_segments": load_notebook_namespace("build_intraminute_segments.ipynb", [1, 2, 3, 4]),
    "intraminute_dynamics": load_notebook_namespace("build_intraminute_dynamics.ipynb", [1, 2, 3, 4]),
    "trades_minute_level": load_notebook_namespace("build_trades_minute_level.ipynb", [1, 2, 3, 4]),
    "trade_distribution": load_notebook_namespace("build_trade_distribution.ipynb", [1, 2, 3, 4]),
    "return_1m_forward": load_notebook_namespace("build_features.ipynb", [0, 2]),
}

sorted(builders)

['aggression_features',
 'intraminute_dynamics',
 'intraminute_features',
 'intraminute_segments',
 'return_1m_forward',
 'trade_distribution',
 'trades_minute_level']

### Execute Targeted Rebuilds

In [7]:
FEATURE_REBUILD_TASKS = [
    {
        "feature": "aggression_features",
        "dates": ["2020-02-01"],
        "builder": lambda ns, date: ns["build_aggression_features_for_day"](SYMBOL, date, s3_client=s3),
    },
    {
        "feature": "intraminute_features",
        "dates": ["2020-02-01"],
        "builder": lambda ns, date: ns["build_intraminute_features_for_day"](SYMBOL, date, s3_client=s3),
    },
    {
        "feature": "intraminute_segments",
        "dates": ["2020-02-01"],
        "builder": lambda ns, date: ns["build_intraminute_segment_features_for_day"](SYMBOL, date, s3_client=s3),
    },
    {
        "feature": "trades_minute_level",
        "dates": ["2020-02-01"],
        "builder": lambda ns, date: ns["build_trades_minute_level_for_day"](SYMBOL, date, s3_client=s3),
    },
    {
        "feature": "trade_distribution",
        "dates": ["2020-02-01"],
        "builder": lambda ns, date: ns["build_trade_distribution_for_day"](SYMBOL, date, s3_client=s3),
    },
    {
        "feature": "intraminute_dynamics",
        "dates": ["2020-02-01", *INTRAMINUTE_DYNAMICS_DAYS_TO_REBUILD],
        "builder": lambda ns, date: ns["build_intraminute_dynamics_for_day"](SYMBOL, date, s3_client=s3),
    },
]

rebuild_rows = []

for task in FEATURE_REBUILD_TASKS:
    feature = task["feature"]
    namespace = builders[feature]
    for date in task["dates"]:
        print(f"rebuild {feature} {date}")
        feature_df = task["builder"](namespace, date)
        key = write_feature_partition(feature_df, feature, date)
        rebuild_rows.append({"feature": feature, "date": date, "rows": len(feature_df), "key": key, "status": "uploaded"})

rebuild_results_df = pd.DataFrame(rebuild_rows)
rebuild_results_df

rebuild aggression_features 2020-02-01
rebuild intraminute_features 2020-02-01
rebuild intraminute_segments 2020-02-01
rebuild trades_minute_level 2020-02-01
rebuild trade_distribution 2020-02-01
rebuild intraminute_dynamics 2020-02-01
rebuild intraminute_dynamics 2020-03-15
rebuild intraminute_dynamics 2022-04-19
rebuild intraminute_dynamics 2022-05-01


,feature,date,rows,key,status
0,aggression_features,2020-02-01,1440,features/aggression_features/symbol=ADAUSDT/in...,uploaded
1,intraminute_features,2020-02-01,1440,features/intraminute_features/symbol=ADAUSDT/i...,uploaded
2,intraminute_segments,2020-02-01,1440,features/intraminute_segments/symbol=ADAUSDT/i...,uploaded
3,trades_minute_level,2020-02-01,1440,features/trades_minute_level/symbol=ADAUSDT/in...,uploaded
4,trade_distribution,2020-02-01,1440,features/trade_distribution/symbol=ADAUSDT/int...,uploaded
5,intraminute_dynamics,2020-02-01,1440,features/intraminute_dynamics/symbol=ADAUSDT/i...,uploaded
6,intraminute_dynamics,2020-03-15,1440,features/intraminute_dynamics/symbol=ADAUSDT/i...,uploaded
7,intraminute_dynamics,2022-04-19,1440,features/intraminute_dynamics/symbol=ADAUSDT/i...,uploaded
8,intraminute_dynamics,2022-05-01,1440,features/intraminute_dynamics/symbol=ADAUSDT/i...,uploaded


### Rebuild `return_1m_forward`

??? `2026-02-01` ???? ?????????? `next_date="2026-02-02"`, ????? ???????????? ?????? `23:59`. ??? `2020-02-01` ????????? ???? ???? ??????????, ?? ????? ? ???????? ??????????????? ?????? `00:00` ???????? ???.

In [8]:
return_namespace = builders["return_1m_forward"]
return_tasks = [
    {"date": "2020-02-01", "next_date": "2020-02-02"},
    {"date": "2026-02-01", "next_date": "2026-02-02"},
]

return_rows = []
for task in return_tasks:
    date = task["date"]
    print(f"rebuild return_1m_forward {date}")
    feature_df = return_namespace["build_forward_return_1m_for_symbol_day"](
        SYMBOL,
        date,
        next_date=task["next_date"],
        s3_client=s3,
    )
    key = write_feature_partition(feature_df, "return_1m_forward", date)
    return_rows.append({"feature": "return_1m_forward", "date": date, "rows": len(feature_df), "key": key, "status": "uploaded"})

return_rebuild_results_df = pd.DataFrame(return_rows)
return_rebuild_results_df

rebuild return_1m_forward 2020-02-01
rebuild return_1m_forward 2026-02-01


,feature,date,rows,key,status
0,return_1m_forward,2020-02-01,1440,features/return_1m_forward/symbol=ADAUSDT/inte...,uploaded
1,return_1m_forward,2026-02-01,1440,features/return_1m_forward/symbol=ADAUSDT/inte...,uploaded


### Unsupported Feature Rebuilds

In [9]:
unsupported_feature_rebuilds = pd.DataFrame(
    [
        {
            "feature": "btc_features",
            "dates": ["2020-02-01"],
            "reason": "No source notebook/formula for btc_features was found in this repository.",
        },
        {
            "feature": "price_pressure",
            "dates": ["2020-02-01"],
            "reason": "No source notebook/formula for price_pressure was found in this repository.",
        },
    ]
)
unsupported_feature_rebuilds

,feature,dates,reason
0,btc_features,[2020-02-01],No source notebook/formula for btc_features wa...
1,price_pressure,[2020-02-01],No source notebook/formula for price_pressure ...


### Verify Rebuilt Feature Partitions

In [10]:
verify_feature_targets = [
    ("aggression_features", "2020-02-01"),
    ("intraminute_features", "2020-02-01"),
    ("intraminute_segments", "2020-02-01"),
    ("trades_minute_level", "2020-02-01"),
    ("trade_distribution", "2020-02-01"),
    ("intraminute_dynamics", "2020-02-01"),
    ("intraminute_dynamics", "2020-03-15"),
    ("intraminute_dynamics", "2022-04-19"),
    ("intraminute_dynamics", "2022-05-01"),
    ("return_1m_forward", "2020-02-01"),
    ("return_1m_forward", "2026-02-01"),
]

feature_verify_df = pd.DataFrame(
    [validate_feature_minutes(feature, date) for feature, date in verify_feature_targets]
)
feature_verify_df

,feature,date,key,rows,unique_minutes,missing_minutes_count,missing_minutes,extra_minutes_count,extra_minutes
0,aggression_features,2020-02-01,features/aggression_features/symbol=ADAUSDT/in...,1440,1440,0,[],0,[]
1,intraminute_features,2020-02-01,features/intraminute_features/symbol=ADAUSDT/i...,1440,1440,0,[],0,[]
2,intraminute_segments,2020-02-01,features/intraminute_segments/symbol=ADAUSDT/i...,1440,1440,0,[],0,[]
3,trades_minute_level,2020-02-01,features/trades_minute_level/symbol=ADAUSDT/in...,1440,1440,0,[],0,[]
4,trade_distribution,2020-02-01,features/trade_distribution/symbol=ADAUSDT/int...,1440,1440,0,[],0,[]
5,intraminute_dynamics,2020-02-01,features/intraminute_dynamics/symbol=ADAUSDT/i...,1440,1440,0,[],0,[]
6,intraminute_dynamics,2020-03-15,features/intraminute_dynamics/symbol=ADAUSDT/i...,1440,1440,0,[],0,[]
7,intraminute_dynamics,2022-04-19,features/intraminute_dynamics/symbol=ADAUSDT/i...,1440,1440,0,[],0,[]
8,intraminute_dynamics,2022-05-01,features/intraminute_dynamics/symbol=ADAUSDT/i...,1440,1440,0,[],0,[]
9,return_1m_forward,2020-02-01,features/return_1m_forward/symbol=ADAUSDT/inte...,1440,1440,0,[],0,[]


### Save Feature Rebuild Reports

In [11]:
report_dir = Path("coverage_reports")
report_dir.mkdir(exist_ok=True)

pd.concat([rebuild_results_df, return_rebuild_results_df], ignore_index=True).to_csv(
    report_dir / "feature_rebuild_results.csv",
    index=False,
)
feature_verify_df.to_csv(report_dir / "feature_rebuild_verify.csv", index=False)
unsupported_feature_rebuilds.to_csv(report_dir / "unsupported_feature_rebuilds.csv", index=False)

print(f"saved: {report_dir / 'feature_rebuild_results.csv'}")
print(f"saved: {report_dir / 'feature_rebuild_verify.csv'}")
print(f"saved: {report_dir / 'unsupported_feature_rebuilds.csv'}")

saved: coverage_reports\feature_rebuild_results.csv
saved: coverage_reports\feature_rebuild_verify.csv
saved: coverage_reports\unsupported_feature_rebuilds.csv
